In [ ]:
config = {
    "vessel_shape": (3,5,3),
    "yard_shape": (3,5,3),
    "num_containers": 50,
    "group_num": 3,
    "num_cranes": 5, 
    "group_placement": "random",
    "seed": 4307,
    "time_penalty_coef": 0.002,
    "env": "SPAEC",
}

In [ ]:
config = {
    "vessel_shape": (5,10,4),
    "yard_shape": (5,10,4),
    "num_containers": 200,
    "group_num": 3,
    "num_cranes": 3, 
    "group_placement": "random",
    "seed": 4307,
    "time_penalty_coef": 0.002,
    "env": "SPAEC",
}

In [ ]:
from envs.stowage_crane_gym import MultiCraneStowageEnv
env = MultiCraneStowageEnv(config=config, render_mode="rgb_array")
obs, info = env.reset()

In [ ]:
env.time_arr

In [ ]:
import matplotlib.pyplot as plt

# Get the RGB array
rgb_array = env.render()

# Display the image
plt.imshow(rgb_array)
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
env.crane_busy_until

In [ ]:
obs, _,terminated,_,info = env.step(500)
print(info["cranes"]["busy_until"]-info["current_time"])
print(env.current_vessel_slots)
print(terminated)

In [ ]:
import numpy as np
for a in list(np.arange(env.action_space.n)[env.action_masks()]):
    print(a, env._decode_action(a))

In [ ]:
from envs.stowage_crane_gym import MultiCraneStowageEnv
from sb3_contrib import MaskablePPO
from algorithms.DQN import MaskableDQN
from algorithms.A2C import MaskableA2C
from algorithms.TRPO import MaskableTRPO
from algorithms.QRDQN import MaskableQRDQN
from sb3_contrib.common.maskable.utils import get_action_masks
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
import numpy as np


class TensorboardCallback(BaseCallback):
    """
    Custom callback for plotting additional values in tensorboard.
    """

    def __init__(self, verbose=0):
        self.is_tb_set = False
        super(TensorboardCallback, self).__init__(verbose)

    def _on_step(self) -> bool:
        if self.num_timesteps % 190 == 0:
            # Log to tensorboard

            self.logger.record("metrics/mean_reward", np.mean(self.locals["rewards"]))
            # print(len(self.locals['infos'][0]['action_sequence']))
            total_shifters_lst = []
            current_time_lst = []
            for i in range(len(self.locals["infos"])):
                total_shifters_lst.append(self.locals["infos"][i].get("total_shifters"))
                current_time_lst.append(self.locals["infos"][i].get("current_time"))
            total_shifters = np.mean(total_shifters_lst)
            current_time = np.mean(current_time_lst)
            self.logger.record("metrics/total_shifters", total_shifters)
            self.logger.record("metrics/current_time", current_time)
            self.logger.dump(self.num_timesteps)
        return True

env = MultiCraneStowageEnv(config=config)
obs, info = env.reset()

model = MaskableQRDQN("MlpPolicy", env, verbose=1, gamma=0.4)
model.learn(10000, callback=TensorboardCallback(), log_interval=4)
# env.reset()
# model = MaskablePPO("MlpPolicy", env, gamma=0.4, verbose=1,tensorboard_log="./multi_cranes/", n_steps=2000)
# model.learn(100,callback=TensorboardCallback(),tb_log_name="PPO_100c_3g_5cr")

In [ ]:
from wandb.integration.sb3 import WandbCallback
from envs.stowage_aec import StowageAEC
import wandb
class TestCallback(WandbCallback):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _on_step(self) -> bool:
        # Call the parent class's method to log the metrics
        super()._on_step()
        if self.num_timesteps % 5000 == 0:
            # Evaluate the policy
            rew_lst = []
            total_shifters_lst = []
            current_time_lst = []
            for i in range(5):
                # env = MultiCraneStowageEnv(config=config)
                env = StowageAEC(config)
                obs, info = env.reset()
                rewards = []
                while True:
                    action, _ = self.model.predict(obs, action_masks=np.expand_dims(env.action_masks(), axis=0)) # Used for MaskableQRDQN
                    # action, _ = self.model.predict(obs, action_masks=env.action_masks())
                    obs, reward, terminated, truncated, info = env.step(action)
                    rewards.append(reward)
                    if terminated or truncated:
                        total_shifters = info.get("total_shifters")
                        current_time = info.get("current_time")
                        break
                mean_r = np.sum(rewards)
                rew_lst.append(mean_r)
                total_shifters_lst.append(total_shifters)
                current_time_lst.append(current_time)
            # Log to tensorboard
            self.logger.record("eval/mean_reward", np.mean(rew_lst))
            self.logger.record("eval/total_shifters", np.mean(total_shifters_lst))
            self.logger.record("eval/current_time", np.mean(current_time_lst))
            self.logger.dump(self.num_timesteps)
        return True


In [10]:
from sb3_contrib import MaskablePPO
from envs.stowage_aec import StowageAEC
from algorithms.DQN import MaskableDQN
from algorithms.A2C import MaskableA2C
from algorithms.TRPO import MaskableTRPO
from algorithms.QRDQN import MaskableQRDQN
from wandb.integration.sb3 import WandbCallback
from sb3_contrib import TRPO
import numpy as np
import wandb
for i in range(1):
    # env = MultiCraneStowageEnv(config=config)
    env = StowageAEC(config)
    obs, info = env.reset()
    run = wandb.init(
        project="sb3",
        config=config,
        sync_tensorboard=True,  # auto-upload sb3's tensorboard metrics
        # monitor_gym=True,  # auto-upload the videos of agents playing the game
        # save_code=True,  # optional
    )
    model = MaskableQRDQN("MlpPolicy", env, verbose=1, tensorboard_log=f"runs/{run.id}", learning_rate=0.0004)
    model.learn(total_timesteps=20000, callback=TestCallback())
    run.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: yunqihuang (alexisloadmaster-loadmaster-ai). Use `wandb login --relogin` to force relogin


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.11/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 9.79GB > 9.47GB
  warnings.warn(


Logging to runs/1zeyq1xb/QRDQN_1
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
det

/home/stowage_env/algorithms/policies.py:447: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  action_masks = th.tensor(action_masks, dtype=th.bool).to(quantiles.device)


deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
deterministic
torch.Size([1, 60])
torch.Size([1, 200, 60])
torch.Size([1, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
torch.Size([32, 60])
torch.Size([32, 200, 60])
torch.Size([32, 200, 60])
deterministic
torch.Size([1, 60])
to

KeyboardInterrupt: 

In [ ]:
obs, _ = env.reset()
while True:
    action, _ = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break